In [56]:
from pathlib import Path
import geopandas as gpd
import joblib
import numpy as np
import pandas as pd


In [57]:
df = pd.read_csv("../data/processed/crmls_sfr_cleaned_base.csv", low_memory=False)
print("Dataset shape:", df.shape)

Dataset shape: (398461, 70)


检查 dtype、缺失率和类别数量
Check columns type

In [58]:
candidate_features = [
    "Latitude",
    "Longitude",
    "City",
    "AssociationFee",
    "AssociationFeeFrequency",
    "HighSchoolDistrict",
    "CloseDate"
]

missing_df = (
    df[candidate_features]
    .isna()
    .mean()
    .to_frame(name="Missing_Percentage (%)")
    .multiply(100)
    .round(2)
)

print(missing_df)

                         Missing_Percentage (%)
Latitude                                   0.07
Longitude                                  0.07
City                                       0.13
AssociationFee                            30.58
AssociationFeeFrequency                   75.18
HighSchoolDistrict                        25.83
CloseDate                                  0.00


In [59]:
dtype = df[candidate_features].dtypes
print(dtype)

Latitude                   float64
Longitude                  float64
City                        object
AssociationFee             float64
AssociationFeeFrequency     object
HighSchoolDistrict          object
CloseDate                   object
dtype: object


- Latitude、Longitude 和 City 数据质量很好，可以继续保留。
- HighSchoolDistrict 可以作为候选，但约四分之一缺失。
- AssociationFee 可以继续调查，不能把缺失值直接当作 0。
- AssociationFeeFrequency 缺失率超过 75%，暂时不加入模型。
- CloseDate 目前是字符串类型，需要转换成日期。

In [60]:
df_engineered = df.copy()


df_engineered["CloseDate"] = pd.to_datetime(
    df_engineered["CloseDate"],
    errors="coerce"
)


numeric_columns = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt"
]

for column in numeric_columns:
    df_engineered[column] = pd.to_numeric(
        df_engineered[column],
        errors="coerce"
    )

创建 PropertyAge

PropertyAge = CloseDate的年份 - YearBuilt

YearBuilt 表示房屋建造年份，但真正影响房屋价值的，通常是房屋在估值时已经使用了多少年。



In [61]:
df_engineered["PropertyAge"] = (
    df_engineered["CloseDate"].dt.year
    - df_engineered["YearBuilt"]
)

df_engineered.loc[
    df_engineered["PropertyAge"] < 0,
    "PropertyAge"
] = np.nan

创建房间和面积比例

In [62]:
df_engineered["BathBedRatio"] = (
    df_engineered["BathroomsTotalInteger"]
    / df_engineered["BedroomsTotal"].where(
        df_engineered["BedroomsTotal"] > 0
    )
)

每间卧室对应的居住面积

In [63]:
df_engineered["LivingAreaPerBedroom"] = (
    df_engineered["LivingArea"]
    / df_engineered["BedroomsTotal"].where(
        df_engineered["BedroomsTotal"] > 0
    )
)

每间浴室对应的居住面积

In [64]:
df_engineered["LivingAreaPerBathroom"] = (
    df_engineered["LivingArea"]
    / df_engineered["BathroomsTotalInteger"].where(
        df_engineered["BathroomsTotalInteger"] > 0
    )
)

土地面积与居住面积比例

In [65]:
df_engineered["LotToLivingRatio"] = (
    df_engineered["LotSizeSquareFeet"]
    / df_engineered["LivingArea"].where(
        df_engineered["LivingArea"] > 0
    )
)

创建 Log 特征

In [66]:
df_engineered["LogLivingArea"] = np.log1p(
    df_engineered["LivingArea"].where(
        df_engineered["LivingArea"] > 0
    )
)

df_engineered["LogLotSize"] = np.log1p(
    df_engineered["LotSizeSquareFeet"].where(
        df_engineered["LotSizeSquareFeet"] > 0
    )
)

创建 AmenityCount

把不同形式的 True 和 False 转换成 1 和 0：

In [67]:
amenity_columns = [
    "PoolPrivateYN",
    "ViewYN",
    "AttachedGarageYN",
    "NewConstructionYN",
    "FireplaceYN"
]

boolean_mapping = {
    True: 1,
    False: 0,
    "True": 1,
    "False": 0,
    "true": 1,
    "false": 0,
    "Yes": 1,
    "No": 0,
    "Y": 1,
    "N": 0,
    1: 1,
    0: 0
}

amenity_data = df_engineered[amenity_columns].replace(boolean_mapping)

amenity_data = amenity_data.apply(
    pd.to_numeric,
    errors="coerce"
)

C:\Users\Tangent\AppData\Local\Temp\ipykernel_28680\2359182500.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  amenity_data = df_engineered[amenity_columns].replace(boolean_mapping)


创建两个特征：

In [68]:
# 已知多少个设施字段
df_engineered["AmenityKnownCount"] = (
    amenity_data.notna().sum(axis=1)
)

# 有多少个设施为 True
df_engineered["AmenityCount"] = (
    amenity_data.sum(
        axis=1,
        min_count=1
    )
)

创建月份周期特征

In [69]:
close_month = df_engineered["CloseDate"].dt.month

df_engineered["CloseMonthSin"] = np.sin(
    2 * np.pi * close_month / 12
)

df_engineered["CloseMonthCos"] = np.cos(
    2 * np.pi * close_month / 12
)

再创建用于时间切分的月份：

In [70]:
df_engineered["split_month"] = (
    df_engineered["CloseDate"].dt.to_period("M")
)

In [71]:
engineered_features = [
    "PropertyAge",
    "BathBedRatio",
    "LivingAreaPerBedroom",
    "LivingAreaPerBathroom",
    "LotToLivingRatio",
    "LogLivingArea",
    "LogLotSize",
    "AmenityCount",
    "AmenityKnownCount",
    "CloseMonthSin",
    "CloseMonthCos"
]

df_engineered[engineered_features].describe().T

,count,mean,std,min,25%,50%,75%,max
PropertyAge,398178.0,48.974408,27.402971,0.000000,27.000000,4.900000e+01,69.000000,2.490000e+02
BathBedRatio,398207.0,0.752952,0.286355,0.000000,0.666667,6.666667e-01,1.000000,8.750000e+01
LivingAreaPerBedroom,398281.0,580.333501,1234.889217,1.000000,445.333333,5.392500e+02,662.750000,7.695600e+05
LivingAreaPerBathroom,398262.0,793.927765,1839.277358,1.000000,650.000000,7.680000e+02,903.250000,1.154340e+06
LotToLivingRatio,391437.0,129.857072,7470.581953,0.000004,3.008215,4.251904e+00,6.014493,1.570930e+06
LogLivingArea,398461.0,7.519359,0.429529,1.098612,7.226209,7.496097e+00,7.791110,1.465219e+01
LogLotSize,391437.0,9.071218,0.947683,0.009950,8.641886,8.888619e+00,9.243388,2.145910e+01
AmenityCount,398440.0,2.186497,1.008600,0.000000,2.000000,2.000000e+00,3.000000,5.000000e+00
AmenityKnownCount,398461.0,4.580478,0.598003,0.000000,4.000000,5.000000e+00,5.000000,5.000000e+00
CloseMonthSin,398461.0,-0.009192,0.712625,-1.000000,-0.866025,-2.449294e-16,0.500000,1.000000e+00


In [72]:
missing_summary = (
    df_engineered[engineered_features]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .to_frame(name="Missing_Percentage (%)")
    .sort_values(by="Missing_Percentage (%)", ascending=False)
)

print(missing_summary)

                       Missing_Percentage (%)
LotToLivingRatio                         1.76
LogLotSize                               1.76
PropertyAge                              0.07
BathBedRatio                             0.06
LivingAreaPerBedroom                     0.05
LivingAreaPerBathroom                    0.05
AmenityCount                             0.01
LogLivingArea                            0.00
AmenityKnownCount                        0.00
CloseMonthSin                            0.00
CloseMonthCos                            0.00


School District GeoJSON


Download the California School District boundary GeoJSON from https://data.ca.gov/dataset/california-school-district-areas-2025-26

In [73]:
DISTRICT_PATH = Path(
    "../data/external/DistrictAreas2526.geojson"
)

districts = gpd.read_file(DISTRICT_PATH)

print("District shape:", districts.shape)
print("District CRS:", districts.crs)

print(
    districts["DistrictType"]
    .value_counts(dropna=False)
)

District shape: (936, 51)
District CRS: EPSG:3857
DistrictType
Elementary    515
Unified       345
High           76
Name: count, dtype: int64


In [74]:
unified_districts = districts[
    districts["DistrictType"] == "Unified"
][
    [
        "DistrictName",
        "CountyName",
        "geometry"
    ]
].copy()

print(
    "Unified district count:",
    len(unified_districts)
)

Unified district count: 345


只保留 Unified District

In [75]:
unified_districts = (
    unified_districts
    .to_crs("EPSG:4326")
)

print(
    "New CRS:",
    unified_districts.crs
)

New CRS: EPSG:4326


将房屋坐标转换成地理点

In [76]:
valid_coordinates = (
    df_engineered["Latitude"].notna()
    & df_engineered["Longitude"].notna()
)

In [77]:
property_points = gpd.GeoDataFrame(
    index=df_engineered.index[valid_coordinates],
    geometry=gpd.points_from_xy(
        df_engineered.loc[
            valid_coordinates,
            "Longitude"
        ],
        df_engineered.loc[
            valid_coordinates,
            "Latitude"
        ]
    ),
    crs="EPSG:4326"
)

print(
    "Property point count:",
    len(property_points)
)
property_points

Property point count: 398174


,geometry
0,POINT (-117.3406 33.14727)
1,POINT (-117.20227 34.23598)
2,POINT (-116.44728 33.78377)
3,POINT (-121.5864 39.76773)
4,POINT (-117.14204 32.97721)
...,...
398456,POINT (-118.18774 33.78687)
398457,POINT (-119.14586 34.37768)
398458,POINT (-118.45237 34.27526)
398459,POINT (-121.96159 37.85465)


执行 Spatial Join

判断每一个房产点位于哪一个 Unified School District polygon 内

In [78]:
district_match = gpd.sjoin(
    property_points,
    unified_districts[
        [
            "DistrictName",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

district_match.head()

,geometry,index_right,DistrictName
0,POINT (-117.3406 33.14727),582.0,Carlsbad Unified
1,POINT (-117.20227 34.23598),533.0,Rim of the World Unified
2,POINT (-116.44728 33.78377),477.0,Palm Springs Unified
3,POINT (-121.5864 39.76773),29.0,Paradise Unified
4,POINT (-117.14204 32.97721),568.0,Poway Unified


检查是否产生重复记录

In [79]:
df_engineered["UnifiedSchoolDistrict"] = (
    district_match["DistrictName"]
    .reindex(df_engineered.index)
)

检查匹配率

In [80]:
valid_coordinate_count = valid_coordinates.sum()

matched_count = (
    df_engineered.loc[
        valid_coordinates,
        "UnifiedSchoolDistrict"
    ]
    .notna()
    .sum()
)

match_rate = (
    matched_count
    / valid_coordinate_count
    * 100
)

print(
    "Valid coordinates:",
    valid_coordinate_count
)

print(
    "Matched Unified districts:",
    matched_count
)

print(
    "Match rate:",
    round(match_rate, 2),
    "%"
)

Valid coordinates: 398174
Matched Unified districts: 297989
Match rate: 74.84 %


In [81]:
df_engineered[
    "UnifiedSchoolDistrict"
].value_counts(
    dropna=False
).head(20)

UnifiedSchoolDistrict
NaN                            100472
Los Angeles Unified             38230
San Diego Unified               10559
Capistrano Unified               7089
Desert Sands Unified             7028
Palm Springs Unified             5980
Oakland Unified                  5170
Corona-Norco Unified             5008
Hemet Unified                    4868
Long Beach Unified               4699
Riverside Unified                4573
Temecula Valley Unified          4419
Mt. Diablo Unified               4155
San Bernardino City Unified      3710
Lake Elsinore Unified            3481
Saddleback Valley Unified        3341
Poway Unified                    3271
Orange Unified                   3065
Hesperia Unified                 3040
Newport-Mesa Unified             3037
Name: count, dtype: int64

In [82]:
output_path = (
    "../data/processed/"
    "crmls_sfr_engineered_full.parquet"
)

df_engineered.to_parquet(
    output_path,
    index=False
)

print(
    "Saved:",
    output_path
)

print(
    "Final shape:",
    df_engineered.shape
)

Saved: ../data/processed/crmls_sfr_engineered_full.parquet
Final shape: (398461, 83)
